## Data preprocessing


In [ ]:
import json
import numpy as np
from gensim.models import KeyedVectors
from gensim.models import Word2Vec
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import pandas as pd
import random
import pickle
from utils import load_data, create_embedding_matrix, preprocess_text, to_padding, pad_sequences
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from sklearn.metrics import precision_recall_fscore_support


In [ ]:
class MyTokenizer:
    def __init__(self):
        self.word_index = {"<PAD>": 0, "<UNK>": 1}
        self.idx_to_token = {0: "<PAD>", 1: "<UNK>"}
        
    def fit_on_texts(self, texts):
        for text in texts:
            for word in text.split():
                if word not in self.word_index:
                    self.word_index[word] = len(self.word_index)
                    self.idx_to_token[self.word_index[word]] = word

    def text_to_sequences(self, text):
        return [self.word_index.get(word, self.word_index["<UNK>"]) for word in text.split()]
    
    def texts_to_sequences(self, texts):
        return [[self.word_index.get(word, self.word_index["<UNK>"]) for word in text.split()] for text in texts]


    def encode(self, text, max_length):
        tokens = self.text_to_sequences(text)
        if len(tokens) < max_length:
            tokens += [self.word_index["<PAD>"]] * (max_length - len(tokens))
        else:
            tokens = tokens[:max_length]
        return tokens
    
    def __call__(self, claims, evidences, max_length=512, return_tensors="pt"):
        self.fit_on_texts([claims, evidences])
        encoded_claims = self.encode(claims, max_length)
        encoded_evidences = self.encode(evidences, max_length)
        if return_tensors == "pt":
            return {
                "input_ids": torch.tensor([encoded_claims, encoded_evidences], dtype=torch.long)
            }
        return {"input_ids": [encoded_claims, encoded_evidences]}


In [ ]:
# nltk.download('punkt')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

train_claims_data = load_data('data/train-claims.json')
evidence_data = load_data('data/evidence.json')
dev_claims_data = load_data('data/dev-claims.json')
evidence_map = load_data('data/curated/preprocessed_evidence_map.json')  

In [ ]:
from sklearn.model_selection import train_test_split

claim_ids = []
for claim_id, claim_details in train_claims_data.items():
	claim_ids.append(claim_id)

# split the claims_df into training and test sets
train, test = train_test_split(claim_ids, test_size=0.2, random_state=42)
len(train)

In [ ]:
# train_data_for_dataframe = []
# test_data_for_dataframe = []
# evidence_keys = list(evidence_map.keys())  # List of all evidence IDs

# for claim_id, claim_details in train_claims_data.items():
# 	claim_text = preprocess_text(claim_details['claim_text'], stemmer, stop_words)
# 	claim_evidences = set(claim_details['evidences'])  # Convert to set for faster checks

# 	# Add positive examples
# 	for eid in claim_evidences:
# 		evidence_text = evidence_map.get(eid, "NULL")  
# 		if evidence_text != "NULL":
# 			data = {
# 				'claim': claim_text,
# 				'evidence': evidence_text,
# 				'label': 1  # Label as relevant
# 			}
# 			if claim_id in train:
# 				train_data_for_dataframe.append(data)
# 			else:
# 				test_data_for_dataframe.append(data)

# 	# Add negative examples
# 	num_neg_samples = min(len(claim_evidences), len(evidence_keys) - len(claim_evidences))  # Limit the number of negative samples
# 	negative_samples = random.sample([k for k in evidence_keys if k not in claim_evidences], num_neg_samples)
# 	for eid in negative_samples:
# 		evidence_text = evidence_map[eid]
# 		data = {
# 			'claim': claim_text,
# 			'evidence': evidence_text,
# 			'label': 0  # Label as not relevant
# 		}
# 		if claim_id in train:
# 				train_data_for_dataframe.append(data)
# 		else:
# 			test_data_for_dataframe.append(data)

# train_df = pd.DataFrame(train_data_for_dataframe)
# test_df = pd.DataFrame(test_data_for_dataframe)

# train_df.to_csv('train_data.csv', index=False)
# test_df.to_csv('test_data.csv', index=False)
train_df = pd.read_csv("train_data.csv")
test_df = pd.read_csv("test_data.csv")

train_df = train_df.dropna()
test_df = test_df.dropna()

print(train_df.head(10))


In [ ]:
tokenizer = MyTokenizer()
x_claim, x_sents, x_labels, x_claims_word_index,  x_sents_word_index, y_claims_data, y_sents_data, y_labels = to_padding(train_df, test_df, tokenizer)

print ("x claim word index ", len(x_claims_word_index))
print ("x sent word index ", len(x_sents_word_index))

vocab_size_claims = len(x_claims_word_index) + 2  # +1 for padding, +1 for <UNK>
vocab_size_evidences = len(x_sents_word_index) + 2

In [ ]:
class EvidenceDataset(Dataset):
    def __init__(self, claims, evidences, labels):
        self.claims = claims
        self.evidences = evidences
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        claim = torch.tensor(self.claims[idx], dtype=torch.long)
        evidence = torch.tensor(self.evidences[idx], dtype=torch.long)
        label = torch.tensor(self.labels[idx], dtype=torch.float)
        return {
            'claims': claim,
            'evidences': evidence,
            'labels': label
        }

# Assuming x_claim, y_claims_data, etc. are numpy arrays or lists of integers
train_dataset = EvidenceDataset(x_claim, x_sents, x_labels)
test_dataset = EvidenceDataset(y_claims_data, y_sents_data, y_labels)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


## Creating the Embedding Matrix

In [ ]:
word_vectors = KeyedVectors.load('word2vec.wordvectors', mmap='r')

In [ ]:
embedding_dim = 300  # dimension of word2vec vectors
(embed_matrix_claim, embed_dim_claim) = create_embedding_matrix(vocab_size_claims, word_vectors, x_claims_word_index, embedding_dim)
(embed_matrix_evidence, embed_dim_evidence) = create_embedding_matrix(vocab_size_evidences, word_vectors, x_sents_word_index, embedding_dim)

print ("embed_matrix_claim shape ", embed_matrix_claim.shape)
print ("embed_matrix_evidence shape ", embed_matrix_evidence.shape)

## Building the LSTM Model

We will build a simple unidirectional LSTM model to compare claim and evidence embeddings.

In [ ]:
class EvidenceModel(nn.Module):
    def __init__(self, vocab_size_claims, vocab_size_evidences, embed_dim_claim, embed_dim_evidence):
        super(EvidenceModel, self).__init__()
        self.embedding_claims = nn.Embedding(vocab_size_claims, embed_dim_claim)
        self.lstm_claims_1 = nn.LSTM(embed_dim_claim, 256, batch_first=True)
        self.lstm_claims_2 = nn.LSTM(256, 16, batch_first=True)
        self.batch_norm_claims = nn.BatchNorm1d(16)
        
        self.embedding_evidences = nn.Embedding(vocab_size_evidences, embed_dim_evidence)
        self.lstm_evidences_1 = nn.LSTM(embed_dim_evidence, 256, batch_first=True)
        self.lstm_evidences_2 = nn.LSTM(256, 64, batch_first=True)
        self.batch_norm_evidences = nn.BatchNorm1d(64)
        
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(16 + 64, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, claims_input, evidences_input):
        embedded_claims = self.embedding_claims(claims_input)
        lstm_out_claims, _ = self.lstm_claims_1(embedded_claims)
        lstm_out_claims, _ = self.lstm_claims_2(lstm_out_claims)
        lstm_out_claims = lstm_out_claims[:, -1, :]  # Get the last output for each sequence
        norm_claims = self.batch_norm_claims(lstm_out_claims)
        
        embedded_evidences = self.embedding_evidences(evidences_input)
        lstm_out_evidences, _ = self.lstm_evidences_1(embedded_evidences)
        lstm_out_evidences, _ = self.lstm_evidences_2(lstm_out_evidences)
        lstm_out_evidences = lstm_out_evidences[:, -1, :]  # Get the last output for each sequence
        norm_evidences = self.batch_norm_evidences(lstm_out_evidences)
        
        concatenated = torch.cat((norm_claims, norm_evidences), dim=1)
        concatenated = self.dropout(concatenated)
        concatenated = self.fc1(concatenated)
        concatenated = self.relu(concatenated)
        output = self.fc2(concatenated)
        output = self.sigmoid(output)
        return output

def train_model(model, train_loader, val_loader, epochs, device):
    model_path = 'lstm_evidence_retrieval.pth'
    best_val_loss = float('inf')
    patience = 2
    trigger_times = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            claims = batch['claims'].to(device)
            evidences = batch['evidences'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            outputs = model(claims, evidences).squeeze(1)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)
        val_loss = evaluate(model, val_loader, device)
        print(f'Epoch {epoch+1}, Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss:.4f}')

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), model_path)
            trigger_times = 0
        else:
            trigger_times += 1
            if trigger_times >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            claims = batch['claims'].to(device)
            evidences = batch['evidences'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(claims, evidences).squeeze(1)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
    return total_loss / len(loader)

def predict(model, data_loader, device):
    model.eval() 
    predictions = []
    with torch.no_grad():  
        for batch in data_loader:
            claims = batch['claims'].to(device)
            evidences = batch['evidences'].to(device)
            outputs = model(claims, evidences)
            predictions.append(outputs.cpu())  # Move predictions to CPU
            
    # Concatenate the list of tensors into a single tensor
    predictions = torch.cat(predictions, dim=0)
    return predictions

def predict_and_filter(model, data_loader, device, threshold=0.5):
    model.eval()
    filtered_claims = []
    filtered_evidences = []
    with torch.no_grad():
        for batch in data_loader:
            claims = batch['claims'].to(device)
            evidences = batch['evidences'].to(device)
            outputs = model(claims, evidences).squeeze(1)
            mask = outputs > threshold  # Get the mask where predictions exceed the threshold

            # Append texts that meet the condition
            filtered_claims.extend([text for text, m in zip(batch['claims_text'], mask) if m])
            filtered_evidences.extend([text for text, m in zip(batch['evidences_text'], mask) if m])

    return filtered_claims, filtered_evidences

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EvidenceModel(vocab_size_claims, vocab_size_evidences, embed_dim_claim, embed_dim_evidence).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

In [ ]:

# # Train the model
# epochs = 60
# train_model(model, train_loader, test_loader, epochs, device)

## Test model

In [ ]:
# Load the best model
model.load_state_dict(torch.load('lstm_evidence_retrieval.pth'))

# Test loader setup like train_loader and val_loader
test_loss = evaluate(model, test_loader, device)
print("Test loss", test_loss)

# Prediction and performance metrics
y_pred = []
y_true = []
model.eval()
with torch.no_grad():
    for batch in test_loader:
        claims = batch['claims'].to(device)
        evidences = batch['evidences'].to(device)
        labels = batch['labels'].cpu().numpy()
        outputs = model(claims, evidences).cpu().numpy().round()
        y_pred.extend(outputs)
        y_true.extend(labels)

# Calculate precision, recall, and F-score
precision, recall, fscore, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')
print("Score of LSTM", {'precision': precision, 'recall': recall, 'fscore': fscore})


## Predict dev-claims

In [ ]:
data_for_dataframe = []
for claim_id, claim_details in dev_claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'], stemmer, stop_words)
    eids = claim_details['evidences']
    data_for_dataframe.append({
			'claim_id': claim_id,
            'claim': claim_text,
            'evidence': eids
        })
    
# Create DataFrame
dev_claims_df = pd.DataFrame(data_for_dataframe)
dev_claims_df 

In [ ]:
print(list(evidence_map)[:10])

In [ ]:
with open('tokenizer_claims.pickle', 'rb') as handle:
	claims_tokenizer = pickle.load(handle)

with open('tokenizer_evidence.pickle', 'rb') as handle:
	sents_tokenizer = pickle.load(handle)

max_claims_length = 70
max_sents_length = 180

In [ ]:
dev_claims = claims_tokenizer.texts_to_sequences(dev_claims_df["claim"].tolist())
dev_sents = sents_tokenizer.texts_to_sequences(evidence_map.values())

dev_claims = pad_sequences(dev_claims, maxlen=max_claims_length)
dev_sents = pad_sequences(dev_sents, maxlen=max_sents_length)
print ("test claims ", dev_claims.shape)
print ("test sents ", dev_sents.shape)

In [ ]:
threshold = 12527

max_out_of_bound = 0
# Assuming test_sents is a numpy array of sequences
for seq in dev_sents:
    if np.any(seq >= threshold):  # Check if any element in the sequence is greater than or equal to the threshold
        # print("Out-of-bounds sequence:", np.max(seq))
        max_out_of_bound = max(np.max(seq), max_out_of_bound)

In [ ]:
class TestData(Dataset):
    def __init__(self, claims, evidences, claims_text, eid):
        self.claims = claims
        self.evidences = evidences
        self.claims_text = claims_text  # Store the original text of the claims
        self.eid = eid  

    def __len__(self):
        return len(self.claims)

    def __getitem__(self, idx):
        claim = torch.tensor(self.claims[idx], dtype=torch.long)
        evidence = torch.tensor(self.evidences[idx], dtype=torch.long)
        claim_text = self.claims_text[idx]
        evidence_text = self.eid[idx]
        return {
            'claims': claim,
            'evidences': evidence,
            'claims_text': claim_text,
            'evidences_text': evidence_text
        }


dev_dataset = TestData(dev_claims, dev_sents, dev_claims_df["claim"].tolist(), ))
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False)


In [ ]:
# Make predictions
model_predictions = predict(model, test_loader, device)

print("Predictions:", model_predictions)


In [ ]:
filtered_claims, filtered_evidences = predict_and_filter(model, test_loader, device, threshold=0.7)
for claim, evidence in zip(filtered_claims, filtered_evidences):
    print("Claim:", claim)
    print("Evidence:", evidence)
    print("----------")